# Best Predict Tutorial

**Best Prediction** estimates a complete 305-day lactation yield from intermittent test-day records. It uses a standard lactation curve as the population baseline and a covariance matrix to project the observed deviations to unmeasured days.

In this tutorial we will create a fake dataset and apply Best Prediction to it. It will consist of the following parts:

1. Apply Best Prediction to one lactation.
2. Apply Best Prediction to a dataframe containing many lactations.
3. Explore settings, including preprocessing and custom column names.
4. Fit your own standard curve and covariance matrix from reference data.
5. Find more information with `help()` and the online documentation.
6. Inspect the standard curve and covariance matrix used by the package.

The examples use the package defaults first, so you can run them without preparing a reference dataset.

Author: Meike van Leerdam

Date: 12-6-2026

First install the lactationcurve package if you have not done this yet with


```bash
pip install lactationcurve
```

In [1]:
# standard packages
import numpy as np
import pandas as pd

# import the functions to be tried out for best predict
from lactationcurve.characteristics.best_predict import (
    COV_MATRIX,
    STANDARD_CURVE,
    best_predict_method,
    best_predict_method_single_lactation,
    fit_autocorrelation_matrix,
    fit_standard_lc,
)

c:\Users\Meike van Leerdam\bovi\.venv\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## Prepare example data set

`TestId` identifies a lactation, `DaysInMilk` is the days since calving at the day of measurement, and `MilkingYield` is the measured 24-hour yield.

In [2]:
dummy_records = []
for test_id, base_yield in [
    (1001, 31.0),
    (1002, 29.5),
    (1003, 28.0),
    (1004, 26.8),
    (1005, 25.5),
]:
    for day_in_milk, adjustment in zip(
        [5, 20, 35, 55, 80, 110, 150, 190, 240, 290],
        [0.0, 2.1, 3.4, 2.6, 1.1, -0.4, -1.7, -2.5, -3.1, -3.6],
    ):
        dummy_records.append(
            {
                "TestId": test_id,
                "DaysInMilk": day_in_milk,
                "MilkingYield": round(base_yield + adjustment + (test_id - 1003) * 0.35, 1),
            }
        )

dummy_df = pd.DataFrame(dummy_records)

reference_df = dummy_df[dummy_df["TestId"].isin([1001, 1002, 1003, 1004])].copy()
target_df = dummy_df[dummy_df["TestId"] == 1005].copy()

dummy_df

,TestId,DaysInMilk,MilkingYield
0,1001,5,30.3
1,1001,20,32.4
2,1001,35,33.7
3,1001,55,32.9
4,1001,80,31.4
5,1001,110,29.9
6,1001,150,28.6
7,1001,190,27.8
8,1001,240,27.2
9,1001,290,26.7


## How Best Prediction works

Best Prediction compares each observed test-day yield with the standard lactation curve. The covariance matrix then projects the observed deviations over the full 305-day grid before the deviations are added to the baseline curve.

The package provides a standard curve and covariance matrix, so the first examples can be run without fitting any.

## 1. Apply Best Prediction to a single lactation

`best_predict_method_single_lactation` works on one lactation at a time. It expects a dataframe containing only that lactation and returns one cumulative 305-day prediction as a float.

In [3]:
single_prediction = best_predict_method_single_lactation(target_df.copy())
print(f"The result of the single lactation curve prediction is: {single_prediction:.3e} kg")

The result of the single lactation curve prediction is: 7.726e+03 kg


## 2. Apply Best Prediction to a full dataframe

We can now predict one lactation, but most datasets contain several lactations. `best_predict_method` is the high-level function: it standardizes the input, groups recording by lactation (`TestId`), predicts each lactation, and returns one result per lactation as a pandas dataframe. When no test-id column is available, all rows are treated as one lactation.

In [4]:
default_predictions = best_predict_method(dummy_df.copy())
default_predictions

,TestId,LactationMilkYield
0,1001,8953.177055
1,1002,8599.612083
2,1003,8264.795805
3,1004,8025.358849
4,1005,7726.062653


## 3. Settings you can change

### Use your own column names

In [5]:
aliased_df = dummy_df.rename(
    columns={
        "TestId": "AnimalId",
        "DaysInMilk": "DIM",
        "MilkingYield": "Yield",
    }
)

aliased_predictions = best_predict_method(
    aliased_df.copy(),
    days_in_milk_col="DIM",
    milking_yield_col="Yield",
    test_id_col="AnimalId",
)

aliased_predictions

,TestId,LactationMilkYield
0,1001,8953.177055
1,1002,8599.612083
2,1003,8264.795805
3,1004,8025.358849
4,1005,7726.062653


## 4. Fit your own standard lactation curve

A standard curve is most useful when it represents the same population as the lactations you want to score. Here we fit a curve from a reference dataframe, estimate its covariance structure, and reuse those fitted ingredients for the target lactation. In production, use a much larger and representative reference population.

In [6]:
fitted_standard_curve = fit_standard_lc(reference_df.copy())
fitted_covariance = fit_autocorrelation_matrix(reference_df.copy(), fitted_standard_curve)["B_hat"]

print("The fitted matrix has the following parameter values:")
print("b1:", fit_autocorrelation_matrix(reference_df.copy(), fitted_standard_curve)["b1"])
print("b2:", fit_autocorrelation_matrix(reference_df.copy(), fitted_standard_curve)["b2"])
print("rho:", fit_autocorrelation_matrix(reference_df.copy(), fitted_standard_curve)["rho"])

fitted_reference_predictions = best_predict_method(
    target_df.copy(),
    standard_lc=fitted_standard_curve,
    covariance_matrix=fitted_covariance,
)

fitted_reference_predictions

The fitted matrix has the following parameter values:
b1: 1.332981071170676e-05
b2: 2.1303782778022606
rho: 0.9925947735916103


,TestId,LactationMilkYield
0,1005,7679.532621


### Fit everything inside `best_predict_method`

If you already have a reference dataframe, the high-level function can fit the covariance matrix for you by setting `fit_standard_lc_from_data=True` and passing `reference_df`. If you want to use a specific lactation curve for a group of cows, e.g. for heifers, if you put only heifers in the `reference_df` the lactation curve will be only based on them. You can do this for any group you would like.

In [7]:
auto_fit_predictions = best_predict_method(
    target_df.copy(),
    fit_standard_lc_from_data=True,
    reference_df=reference_df.copy(),
)

auto_fit_predictions

,TestId,LactationMilkYield
0,1005,7684.051893


### Pass your own standard curve and covariance matrix directly

You can also provide your own curve and covariance matrix directly when they have already been fitted or supplied by another workflow.

In [8]:
custom_standard_curve = STANDARD_CURVE * 0.98
custom_covariance_matrix = COV_MATRIX.copy()
custom_covariance_matrix = (
    custom_covariance_matrix * 0.9 + np.eye(custom_covariance_matrix.shape[0]) * 0.1
)

custom_prediction = best_predict_method_single_lactation(
    target_df.copy(),
    standard_lc=custom_standard_curve,
    covariance_matrix=custom_covariance_matrix,
)

custom_prediction

np.float64(7723.071781076493)

## 5. Learn more with `help()` and documentation

Python's `help()` function displays the package docstring, including parameters, return values, and important notes. In VS Code or Jupyter, you can also use `?` after a function name.

In [9]:
help(best_predict_method)

Help on function best_predict_method in module lactationcurve.characteristics.best_predict:

best_predict_method(df: pandas.core.frame.DataFrame, standard_lc: numpy.ndarray = STANDARD_CURVE, days_in_milk_col: str | None = None, milking_yield_col: str | None = None, test_id_col: str | None = None, default_test_id: int = 0, covariance_matrix: numpy.ndarray | None = COV_MATRIX, fit_standard_lc_from_data: bool = False, reference_df: pandas.core.frame.DataFrame | None = None) -> pandas.core.frame.DataFrame
    Apply best prediction to one or more lactations.

    By default this function uses the package-provided standard curve and covariance matrix.
    But it is also possible to provide your own standard curve and covariance matrix,
    for example when you want to fit these ingredients from your own reference population.
    This can be done in two ways: either by fitting the covariance matrix and standard curve
    directly from a reference dataset by providing a pandas dataframe at 're

For the full API reference, see the [lactationcurve documentation](https://bovi-analytics.github.io/bovi/lactationcurve/characteristics/best_predict.html) and the [Best Prediction source module](https://github.com/Bovi-analytics/bovi/blob/main/packages/models/lactationcurve/src/lactationcurve/characteristics/best_predict.py).

## 6. Inspect the standard curve and covariance matrix

The package currently uses one standard lactation curve with its corresponding covariance matrix. In your own workflow, you can fit these assets for a relevant subpopulation and pass them to the prediction functions.

In [13]:
import matplotlib.pyplot as plt

days_in_milk = np.arange(1, len(STANDARD_CURVE) + 1)

plt.figure(figsize=(10, 5))
plt.plot(days_in_milk, STANDARD_CURVE)
plt.xlabel("Days in Milk")
plt.ylabel("Milking Yield")
plt.title("Wood Standard Lactation Curve in package lactationcurve")
plt.grid(True)
plt.show()

<Figure size 1000x500 with 1 Axes>

In [11]:
print("The covariance matrix currently used in the best predict method is:")
print(COV_MATRIX)

The covariance matrix currently used in the best predict method is:
[[75.16786  67.28662  67.05099  ... 23.40674  23.32477  23.243088]
 [67.28662  75.16786  67.28662  ... 23.488997 23.40674  23.32477 ]
 [67.05099  67.28662  75.16786  ... 23.571543 23.488997 23.40674 ]
 ...
 [23.40674  23.488997 23.571543 ... 75.16786  67.28662  67.05099 ]
 [23.32477  23.40674  23.488997 ... 67.28662  75.16786  67.28662 ]
 [23.243088 23.32477  23.40674  ... 67.05099  67.28662  75.16786 ]]


## What you learned

- `best_predict_method_single_lactation` predicts one 305-day lactation yield.
- `best_predict_method` applies the same calculation to every lactation in a dataframe.
- The package defaults provide the standard curve and covariance matrix for a quick start.
- Reference data can be used to fit population-specific lactation curve.
- Custom column names and directly supplied assets let you integrate Best Prediction into an existing data pipeline.